# SEC EDGAR Raw Data Test

**Phase 1 — No cloud, no Snowflake. Pure HTTP → DataFrame.**

Tests all 3 public SEC EDGAR endpoints:
1. `company_tickers_exchange.json` — full ticker/exchange snapshot
2. `submissions/CIK{cik}.json` — company metadata (SIC, entity type, filings list)
3. `api/xbrl/companyfacts/CIK{cik}.json` — XBRL financial facts

**Mandatory SEC compliance:**
- `User-Agent` header on every request (SEC Developer Resources requirement)
- Rate limited to ≤8 req/s (SEC hard limit is 10 req/s per IP)

In [1]:
# ── 0. Dependencies (uv) ─────────────────────────────────────────────────────
# From the repo root in PowerShell (run once):
#
#   uv sync --group dev
#   uv run jupyter notebook notebooks\sec_edgar_raw_test.ipynb
#
# uv reads pyproject.toml + .python-version and creates .venv automatically.
# No Activate.ps1 needed -- `uv run` handles the venv.
#
# Install uv on Windows (once, system-wide):
#   powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

In [2]:
import os, json, time, threading, sys
from datetime import date, datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from requests.adapters import HTTPAdapter, Retry
import pandas as pd

# ── SEC User-Agent (required by SEC Developer Resources) ─────────────────────
os.environ["SEC_USER_AGENT"] = "n/a prototyping paul.ananth@yahoo.com"

USER_AGENT = os.environ["SEC_USER_AGENT"]
print(f"User-Agent : {USER_AGENT}")
print(f"Date       : {date.today()}")

User-Agent : n/a prototyping paul.ananth@yahoo.com
Date       : 2026-04-07


In [3]:
# ── Rate limiter (token-bucket, 8 req/s) ─────────────────────────────────────
class RateLimiter:
    """Thread-safe token-bucket. Caps at 8 req/s (20% below SEC's 10 req/s limit)."""
    def __init__(self, rps=8.0):
        self._rps, self._tokens, self._last = rps, rps, time.monotonic()
        self._lock = threading.Lock()
    def acquire(self):
        with self._lock:
            now = time.monotonic()
            self._tokens = min(self._rps, self._tokens + (now - self._last) * self._rps)
            self._last = now
            if self._tokens < 1.0:
                time.sleep((1.0 - self._tokens) / self._rps)
                self._tokens = 0.0
            else:
                self._tokens -= 1.0

# ── HTTP session (User-Agent injected on every request) ──────────────────────
_retry = Retry(total=3, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503])
_session = requests.Session()
_session.headers["User-Agent"] = USER_AGENT
_session.mount("https://", HTTPAdapter(max_retries=_retry))

_limiter = RateLimiter(rps=8.0)

def fetch(url):
    """Rate-limited GET. Returns parsed JSON or None on 404."""
    _limiter.acquire()
    r = _session.get(url, timeout=20)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    return r.json()

print("HTTP client ready  (User-Agent set, rate limiter 8 req/s)")

HTTP client ready  (User-Agent set, rate limiter 8 req/s)


---
## 1. Ticker Exchange Snapshot
Single daily file — all tickers listed on NYSE and Nasdaq (~10,000 rows).

In [4]:
raw = fetch("https://www.sec.gov/files/company_tickers_exchange.json")

tickers_df = pd.DataFrame(raw["data"], columns=raw["fields"])
tickers_df["cik"] = tickers_df["cik"].astype(str).str.zfill(10)

print(f"Total rows : {len(tickers_df):,}")
print(f"Exchanges  : {tickers_df['exchange'].value_counts().to_dict()}")
tickers_df.head(10)

Total rows : 10,433
Exchanges  : {'Nasdaq': 4248, 'NYSE': 3273, 'OTC': 2596, 'CBOE': 28}


,cik,name,ticker,exchange
0,0001045810,NVIDIA CORP,NVDA,Nasdaq
1,0000320193,Apple Inc.,AAPL,Nasdaq
2,0001652044,Alphabet Inc.,GOOGL,Nasdaq
3,0000789019,MICROSOFT CORP,MSFT,Nasdaq
4,0001018724,AMAZON COM INC,AMZN,Nasdaq
5,0001730168,Broadcom Inc.,AVGO,Nasdaq
6,0001318605,"Tesla, Inc.",TSLA,Nasdaq
7,0001326801,"Meta Platforms, Inc.",META,Nasdaq
8,0001067983,BERKSHIRE HATHAWAY INC,BRK-B,NYSE
9,0000104169,Walmart Inc.,WMT,Nasdaq


In [5]:
# Filter to NYSE + Nasdaq only
listed = tickers_df[tickers_df["exchange"].isin(["NYSE", "Nasdaq"])].copy()
print(f"NYSE + Nasdaq : {len(listed):,} tickers")
listed.sample(10)

NYSE + Nasdaq : 7,521 tickers


,cik,name,ticker,exchange
5655,0000862692,CRYO CELL INTERNATIONAL INC,CCEL,NYSE
1051,0000314590,SASOL LTD,SSL,NYSE
1534,0000090896,"Champion Homes, Inc.",SKY,NYSE
3300,0001809691,AMTD Digital Inc.,HKD,NYSE
7540,0001652044,Alphabet Inc.,GOOG,Nasdaq
4685,0001577445,Odysight.ai Inc.,ODYS,Nasdaq
8682,0001042046,AMERICAN FINANCIAL GROUP INC,AFGE,NYSE
5651,0000813298,"DESTINATION XL GROUP, INC.",DXLG,Nasdaq
3948,0002007596,"TWFG, Inc.",TWFG,Nasdaq
2366,0001596993,DORIAN LPG LTD.,LPG,NYSE


---
## 2. Submissions — Full Company Record (ALL fields)

One JSON file per CIK. Contains every field the SEC stores about the registrant:
company metadata, addresses, former names, fiscal year end, insider transaction flags —
plus a full filings index (`filings.recent`) stored as columnar parallel arrays.

**No fields are dropped.** `Submission` Pydantic model maps every top-level key.
`filings.recent` is exploded from columnar format → individual `Filing` rows.

In [6]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("../scripts/ingest").resolve()))

from models import Submission, explode_filings

# Pilot set — 10 well-known companies
PILOT = [
    ("AAPL",  "0000320193"),
    ("MSFT",  "0000789019"),
    ("AMZN",  "0001018724"),
    ("GOOG",  "0001652044"),
    ("META",  "0001326801"),
    ("TSLA",  "0001318605"),
    ("JPM",   "0000019617"),
    ("JNJ",   "0000200406"),
    ("XOM",   "0000034088"),
    ("BRK-B", "0001067983"),
]

submissions_meta = []   # one row per company (all top-level fields)
all_filings      = []   # one row per filing (filings.recent exploded)

for ticker, cik10 in PILOT:
    raw = fetch(f"https://data.sec.gov/submissions/CIK{cik10}.json")
    if raw is None:
        print(f"  {ticker:<8} 404")
        continue

    # Validate the full response — raises ValidationError if SEC changes the schema
    sub = Submission.model_validate(raw)

    # ── Company-level metadata (all top-level fields) ────────────────────────
    meta = sub.model_dump(exclude={"filing_files"})
    meta["ticker"]         = ticker
    meta["filing_files_count"] = len(sub.filing_files)
    # Flatten addresses to individual columns
    for side in ("mailing", "business"):
        addr = (sub.addresses or {}).get(side)
        if addr:
            for k, v in addr.model_dump().items():
                meta[f"addr_{side}_{k}"] = v
    del meta["addresses"]   # replaced by flat columns above
    submissions_meta.append(meta)

    # ── Filings index (filings.recent exploded to rows) ──────────────────────
    recent = raw.get("filings", {}).get("recent", {})
    filing_rows = explode_filings(cik10, recent)
    for r in filing_rows:
        r["ticker"] = ticker
    all_filings.extend(filing_rows)

    print(f"  {ticker:<8} {sub.name[:45]:<45}  "
          f"fisc_yr_end={sub.fiscalYearEnd}  "
          f"filings={len(filing_rows)}")

submissions_df = pd.DataFrame(submissions_meta)
filings_df     = pd.DataFrame(all_filings)

print(f"\nsubmissions_df : {submissions_df.shape[0]} rows × {submissions_df.shape[1]} columns")
print(f"filings_df     : {filings_df.shape[0]} rows × {filings_df.shape[1]} columns")

ValidationError: 3 validation errors for Submission
formerNames.0.date
  Field required [type=missing, input_value={'name': 'APPLE INC', 'fr...19-08-05T04:00:00.000Z'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
formerNames.1.date
  Field required [type=missing, input_value={'name': 'APPLE COMPUTER ...07-01-04T05:00:00.000Z'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
formerNames.2.date
  Field required [type=missing, input_value={'name': 'APPLE COMPUTER ...97-07-28T04:00:00.000Z'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [ ]:
# Company metadata — all fields
print("Columns:", submissions_df.columns.tolist())
submissions_df

In [ ]:
# Filings index — one row per filing (accessionNumber, form, isXBRL, size, primaryDocument)
print(f"Total filings across {len(PILOT)} companies: {len(filings_df):,}")
filings_df[["ticker", "cik", "accessionNumber", "filingDate", "form",
            "isXBRL", "isInlineXBRL", "size", "primaryDocument"]].head(20)

---
## 3. XBRL Company Facts — ALL Concepts, ALL Namespaces (Raw)

The `companyfacts` endpoint contains every tagged financial value ever filed.
A large company (AAPL) has ~3,000+ XBRL concepts across three namespaces:
- **`us-gaap`** — US GAAP financial statement items
- **`dei`** — Document & Entity Information (shares outstanding, fiscal year end, etc.)
- **`ifrs-full`** — IFRS equivalents (non-US filers)

**No filtering.** Every namespace, every concept, every unit, every entry is kept.
The `frame` column (e.g. `CY2023`, `CY2023Q3I`) is preserved — it identifies
SEC-standardised calendar-period tags useful for cross-company comparison.

In [ ]:
from models import CompanyFacts, explode_facts

all_facts = []

# Sequential fetch — companyfacts payloads are 5–15 MB each
for ticker, cik10 in PILOT:
    raw = fetch(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json")
    if raw is None:
        print(f"  {ticker:<8} 404")
        continue

    # Validate full structure via Pydantic
    parsed = CompanyFacts.model_validate(raw)

    # Explode ALL namespaces / concepts / entries — no filter
    rows = explode_facts(parsed)
    for r in rows:
        r["ticker"] = ticker
    all_facts.extend(rows)

    namespaces = list(parsed.facts.keys())
    concepts_per_ns = {ns: len(c) for ns, c in parsed.facts.items()}
    print(f"  {ticker:<8}  {len(rows):>6,} rows  |  namespaces: {namespaces}  |  concepts: {concepts_per_ns}")

facts_df = pd.DataFrame(all_facts)
facts_df["val"]   = pd.to_numeric(facts_df["val"], errors="coerce")
facts_df["end"]   = pd.to_datetime(facts_df["end"], errors="coerce")
facts_df["filed"] = pd.to_datetime(facts_df["filed"], errors="coerce")
# start is None for instant facts — keep as object column
print(f"\nTotal raw fact rows : {len(facts_df):,}")
print(f"Columns             : {facts_df.columns.tolist()}")

In [ ]:
facts_df.head(10)

---
## 4. Spot Checks — What's in the Raw Data?

In [ ]:
# Namespace breakdown — how many rows per namespace?
print("Rows by namespace:")
print(facts_df["namespace"].value_counts().to_string())
print()

# Concept count per namespace
for ns, grp in facts_df.groupby("namespace"):
    print(f"  {ns:<12}  {grp['concept'].nunique():>5} unique concepts  |  {len(grp):>8,} rows")

In [ ]:
# Annual revenue — filter in the DataFrame using concept name + namespace
# (no hardcoded list needed — query by name just like you would in DuckDB)
REVENUE_CONCEPTS = ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"]

revenue = (
    facts_df[
        (facts_df["namespace"] == "us-gaap") &
        (facts_df["concept"].isin(REVENUE_CONCEPTS)) &
        (facts_df["unit"] == "USD") &
        (facts_df["form"].isin(["10-K", "10-K/A"])) &
        (facts_df["start"].notna())   # annual = has a start date
    ]
    .assign(days=lambda x: (x["end"] - pd.to_datetime(x["start"])).dt.days)
    .query("355 <= days <= 375")      # annual period
    .sort_values("end", ascending=False)
    .groupby("ticker").head(5)
    [["ticker", "end", "val", "form", "filed", "frame"]]
    .assign(revenue_bn=lambda x: (x["val"] / 1e9).round(2))
)
revenue

In [ ]:
# Revenue pivot by fiscal year
pivot = (
    revenue
    .assign(year=revenue["end"].dt.year)
    .drop_duplicates(["ticker", "year"])
    .pivot_table(index="ticker", columns="year", values="revenue_bn", aggfunc="first")
    .sort_index()
)
print("Annual Revenue ($B)")
pivot

In [ ]:
# DEI namespace — shares outstanding and entity info (previously dropped entirely)
dei_df = facts_df[facts_df["namespace"] == "dei"].copy()
print(f"DEI rows: {len(dei_df):,}")
print("\nDEI concepts available:")
print(dei_df["concept"].value_counts().head(20).to_string())

---
## 5. Data Quality Summary

In [ ]:
# Submissions metadata field coverage
print("Submissions metadata field coverage:")
for col in submissions_df.columns:
    filled = submissions_df[col].notna().sum()
    print(f"  {col:<45} {100*filled/len(submissions_df):5.1f}%")

In [ ]:
# Overall summary
print("=" * 60)
print("PHASE 1 — RAW DATA SUMMARY (all fields, no filtering)")
print("=" * 60)
print(f"  Tickers snapshot   : {len(tickers_df):>8,} rows")
print(f"  NYSE + Nasdaq      : {len(listed):>8,} tickers")
print()
print(f"  Submissions        : {len(submissions_df):>8} companies  x {submissions_df.shape[1]} columns")
print(f"  Filings index      : {len(filings_df):>8,} filing rows x {filings_df.shape[1]} columns")
print()
print(f"  Facts total        : {len(facts_df):>8,} rows")
print(f"  Unique concepts    : {facts_df['concept'].nunique():>8,}")
print(f"  Namespaces         : {sorted(facts_df['namespace'].unique())}")
print(f"  With frame tag     : {facts_df['frame'].notna().sum():>8,} rows")
print(f"  USD facts          : {(facts_df['unit']=='USD').sum():>8,} rows")
print()
print("SEC compliance:")
print(f"  User-Agent         : {USER_AGENT}")
print(f"  Rate limit         : 8 req/s  (SEC limit = 10 req/s per IP)")